Our prompt needs to assist users in writing three specific types of output for AWS use cases:

- Python code
- JSON configuration files
- Regular expressions

In [ ]:
from typing import Any

from anthropic import Anthropic
from anthropic.types import MessageParam
from dotenv import load_dotenv

load_dotenv()

Messages = list[MessageParam]

client: Anthropic = Anthropic()
model: str = "claude-sonnet-4-6"

In [ ]:
def add_user_message(messages: Messages, text: str) -> None:
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages: Messages, text: str) -> None:
    messages.append({"role": "assistant", "content": text})

def chat(messages: Messages, system_prompt: str|None = None, temperature: float = 1.0) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system_prompt is not None:
        params["system"] = system_prompt
    
    response = client.messages.create(**params)
    return response.content[0].text

In [ ]:
import json

def generate_dataset() -> list[dict[str, str]]:
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages: Messages = []
    add_user_message(messages, prompt)
    text = chat(messages, system_prompt="Respond only with raw valid JSON. No explanation, no markdown, no code blocks.")
    return json.loads(text.strip())

In [ ]:
dataset = generate_dataset()

with open("dataset.local.json", "w") as f:
    json.dump(dataset, f, indent=2)


In [ ]:
def run_prompt(test_case: dict[str, str]) -> str:
    """Merges the prompt and test case input, then returns the result."""
    
    prompt = f""""
Please, solve the following task:

{test_case["task"]}
    """
    messages: Messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case: dict[str, str]) -> dict[str, Any]:
    """Calls run_prompt, then grades the result."""
    
    output = run_prompt(test_case)
    
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [ ]:
def run_eval(dataset: list[dict[str, str]]) -> list[dict[str, Any]]:
    """Loads the dataset and calls run_test_case with each case."""

    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [ ]:
with open("dataset.local.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [ ]:
print(json.dumps(results, indent=2))